In [ ]:
import numpy as np
import pandas as pd
import joblib
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix, f1_score, recall_score
from sklearn.ensemble import RandomForestClassifier
import xgboost as xgb

# 1. Charger les données prétraitées
data = joblib.load('../new_models/processed_data.joblib')

X_train = data['X_train']
X_test = data['X_test']
y_train = data['y_train']
y_test = data['y_test']
feature_names = data['feature_names']

print(f"Données d'entraînement prêtes : {X_train.shape[0]} lignes, {X_train.shape[1]} colonnes.")

In [ ]:
# Initialisation du modèle Random Forest
rf_model = RandomForestClassifier(
    n_estimators=200,
    max_depth=5,                 # Contrainte de profondeur pour éviter l'overfitting
    min_samples_leaf=5,
    class_weight='balanced_subsample', # Équilibrage des classes
    random_state=42,
    n_jobs=-1
)

# Entraînement
rf_model.fit(X_train, y_train)

# Calcul des probabilités de churn sur le test set
y_proba_rf = rf_model.predict_proba(X_test)[:, 1]
print("Random Forest entraîné avec succès.")

In [ ]:
# Calcul du ratio pour pondérer XGBoost
scale_pw = (y_train == 0).sum() / (y_train == 1).sum()

# Initialisation du modèle XGBoost
xgb_model = xgb.XGBClassifier(
    n_estimators=150,
    max_depth=4,                  # Arbres courts = meilleure généralisation
    learning_rate=0.05,
    scale_pos_weight=scale_pw,    # Gestion du déséquilibre de classe
    random_state=42,
    eval_metric='logloss'
)

# Entraînement
xgb_model.fit(X_train, y_train)

# Calcul des probabilités de churn sur le test set
y_proba_xgb = xgb_model.predict_proba(X_test)[:, 1]
print("XGBoost entraîné avec succès.")

In [ ]:
# Choisir un seuil adapté au déséquilibre métier (ex: 25%)
SEUIL = 0.25

# Application du seuil personnalisé sur les probabilités
y_pred_rf_custom = (y_proba_rf >= SEUIL).astype(int)
y_pred_xgb_custom = (y_proba_xgb >= SEUIL).astype(int)

# 1. Évaluation du Random Forest
print("======== EVALUATION RANDOM FOREST (SEUIL 0.25) ========")
print(classification_report(y_test, y_pred_rf_custom))

# 2. Évaluation de XGBoost
print("\n======== EVALUATION XGBOOST (SEUIL 0.25) ========")
print(classification_report(y_test, y_pred_xgb_custom))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Matrice Random Forest
cm_rf = confusion_matrix(y_test, y_pred_rf_custom)
sns.heatmap(cm_rf, annot=True, fmt='d', cmap='Blues', ax=axes[0])
axes[0].set_title(f"Matrice de Confusion - Random Forest (Seuil {SEUIL})")
axes[0].set_xlabel("Prédictions")
axes[0].set_ylabel("Vrais Labels")

# Matrice XGBoost
cm_xgb = confusion_matrix(y_test, y_pred_xgb_custom)
sns.heatmap(cm_xgb, annot=True, fmt='d', cmap='Greens', ax=axes[1])
axes[1].set_title(f"Matrice de Confusion - XGBoost (Seuil {SEUIL})")
axes[1].set_xlabel("Prédictions")
axes[1].set_ylabel("Vrais Labels")

plt.tight_layout()
plt.show()

In [ ]:
# Supposons que XGBoost soit le gagnant grâce à sa structure séquentielle
joblib.dump(xgb_model, '../new_models/best_churn_model.joblib')
print("Le modèle final candidat a été sauvegardé avec succès.")